<a href="https://colab.research.google.com/github/naina-deep/Amazon-performance/blob/main/Copy_of_PRACTICAL_GITHUB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving amazon_data.csv to amazon_data (1).csv


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("AmazonAnalysis").getOrCreate()

df = spark.read.csv("amazon_data.csv", header=True, inferSchema=True)
df.show()

+---+-----------------+-----------+------+-----+------+-------+------+----------+--------+
| ID|     Product_Name|   Category| Price|Stock|Rating|Reviews| Brand|Date_Added|Discount|
+---+-----------------+-----------+------+-----+------+-------+------+----------+--------+
|  1|          Oil Son|     Sports| 64.55|   95|   2.1|    228|BrandB|17-07-2024|   33.83|
|  2|        Push Term|       Toys|  52.6|   55|   1.1|     95|BrandB|11-05-2024|   25.27|
|  3|     Resource Sit|Electronics|285.01|   92|   3.6|    558|BrandD|18-05-2024|   22.46|
|  4|     Only Citizen|      Books|406.62|    1|   4.0|    163|BrandD|17-03-2024|   13.89|
|  5|          It Once|   Clothing|479.03|   44|   1.4|    389|BrandA|08-03-2024|   42.37|
|  6|        That Most|       Toys|139.62|    6|   3.9|    549|BrandA|27-02-2024|    3.94|
|  7|       Find Sound|      Books|416.41|   80|   4.5|    370|BrandE|01-06-2024|   35.23|
|  8|    Month Whether|Electronics|334.02|   99|   2.2|     81|BrandB|18-07-2024|   19.01|

In [ ]:
df.printSchema()
df.show(5)

root
 |-- ID: integer (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Stock: integer (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Date_Added: string (nullable = true)
 |-- Discount: double (nullable = true)

+---+------------+-----------+------+-----+------+-------+------+----------+--------+
| ID|Product_Name|   Category| Price|Stock|Rating|Reviews| Brand|Date_Added|Discount|
+---+------------+-----------+------+-----+------+-------+------+----------+--------+
|  1|     Oil Son|     Sports| 64.55|   95|   2.1|    228|BrandB|17-07-2024|   33.83|
|  2|   Push Term|       Toys|  52.6|   55|   1.1|     95|BrandB|11-05-2024|   25.27|
|  3|Resource Sit|Electronics|285.01|   92|   3.6|    558|BrandD|18-05-2024|   22.46|
|  4|Only Citizen|      Books|406.62|    1|   4.0|    163|BrandD|17-03-2024|   13.89|
|

In [ ]:
df = df.withColumn("Demand_Score", col("Rating") * col("Reviews"))

In [ ]:
df.groupBy("Category").avg("Price").show()

+-----------+------------------+
|   Category|        avg(Price)|
+-----------+------------------+
|       Home|258.20404494382024|
|     Sports|  262.605934065934|
|Electronics| 272.5128985507247|
|   Clothing|262.02384615384614|
|      Books|243.67765432098764|
|       Toys|237.31076086956517|
+-----------+------------------+



In [ ]:
df.orderBy(desc("Demand_Score")).select("Category","Price","Rating","Reviews","Demand_Score").show()

+-----------+------+------+-------+------------------+
|   Category| Price|Rating|Reviews|      Demand_Score|
+-----------+------+------+-------+------------------+
|       Toys|249.27|   5.0|    980|            4900.0|
|Electronics|138.47|   4.7|    971|            4563.7|
|      Books|383.63|   4.8|    944|            4531.2|
|   Clothing|205.25|   4.5|    990|            4455.0|
|       Home|194.07|   4.9|    895|            4385.5|
|       Toys| 57.98|   5.0|    856|            4280.0|
|       Home|391.79|   4.6|    926| 4259.599999999999|
|     Sports|276.16|   4.3|    977| 4201.099999999999|
|       Home|255.89|   4.7|    893|            4197.1|
|     Sports| 228.1|   4.2|    990|            4158.0|
|       Home|109.14|   4.6|    903| 4153.799999999999|
|   Clothing|368.85|   4.2|    983|            4128.6|
|     Sports|114.71|   4.2|    982| 4124.400000000001|
|   Clothing|176.97|   4.7|    864|            4060.8|
|Electronics| 279.5|   4.6|    871|            4006.6|
|   Clothi

In [ ]:
df.groupBy("Brand").avg("Rating").show()

+------+------------------+
| Brand|       avg(Rating)|
+------+------------------+
|BrandE|2.9894736842105267|
|BrandC|2.9787037037037045|
|BrandD|2.9650485436893192|
|BrandB| 2.878021978021977|
|BrandA|  3.15145631067961|
+------+------------------+



In [ ]:
from pyspark.sql.window import Window
windowSpec = Window.partitionBy("Category").orderBy(desc("Rating"))
df.withColumn("Rank", rank().over(windowSpec)).show()

+---+-----------------+--------+------+-----+------+-------+------+----------+--------+------------------+----+
| ID|     Product_Name|Category| Price|Stock|Rating|Reviews| Brand|Date_Added|Discount|      Demand_Score|Rank|
+---+-----------------+--------+------+-----+------+-------+------+----------+--------+------------------+----+
|151|        He Second|   Books|404.07|   60|   4.9|    144|BrandB|27-09-2023|   36.21|             705.6|   1|
|190|      Under Often|   Books| 66.43|    3|   4.9|    175|BrandE|16-01-2024|    41.6| 857.5000000000001|   1|
|212|        Green End|   Books|341.42|   31|   4.9|    625|BrandC|22-01-2024|    11.7|            3062.5|   1|
|259|Population Reduce|   Books|369.76|   45|   4.9|    133|BrandC|19-12-2023|   28.08|             651.7|   1|
| 20|     Society Girl|   Books|383.63|   70|   4.8|    944|BrandE|07-09-2024|   29.95|            4531.2|   5|
| 41|      Along There|   Books|234.04|   97|   4.7|      9|BrandD|25-10-2023|   49.92|42.30000000000000

In [ ]:
df = df.withColumn("Date_Added", to_date(col("Date_Added")))
df = df.withColumn("Month", month("Date_Added"))


In [ ]:
df.groupBy("Category").count().show()

+-----------+-----+
|   Category|count|
+-----------+-----+
|       Home|   89|
|     Sports|   91|
|Electronics|   69|
|   Clothing|   78|
|      Books|   81|
|       Toys|   92|
+-----------+-----+



In [ ]:
df.orderBy(desc("Price")).select("Category","Price","Brand").show()

+-----------+------+------+
|   Category| Price| Brand|
+-----------+------+------+
|     Sports|498.61|BrandD|
|     Sports|497.76|BrandB|
|      Books|494.87|BrandC|
|   Clothing|494.52|BrandE|
|   Clothing|493.89|BrandA|
|Electronics|493.57|BrandE|
|   Clothing|492.99|BrandC|
|Electronics| 492.8|BrandD|
|      Books|491.37|BrandA|
|     Sports| 490.6|BrandE|
|       Home|490.32|BrandD|
|     Sports|487.37|BrandB|
|       Home|487.18|BrandB|
|       Toys|486.21|BrandE|
|       Home|484.84|BrandC|
|       Home|484.68|BrandD|
|     Sports|484.68|BrandC|
|       Home|483.96|BrandD|
|       Home|483.74|BrandA|
|       Toys| 483.3|BrandD|
+-----------+------+------+
only showing top 20 rows


In [ ]:
df.orderBy("Price").select("Category","Price","Brand").show()

+-----------+-----+------+
|   Category|Price| Brand|
+-----------+-----+------+
|       Home|12.18|BrandA|
|     Sports|12.72|BrandA|
|Electronics|13.77|BrandB|
|Electronics|13.79|BrandD|
|      Books| 16.2|BrandD|
|   Clothing|17.28|BrandB|
|      Books|17.67|BrandD|
|Electronics|19.31|BrandE|
|       Home|19.54|BrandC|
|Electronics|20.68|BrandD|
|       Home|21.36|BrandB|
|       Home|21.72|BrandA|
|       Home|22.88|BrandE|
|       Toys|23.82|BrandE|
|      Books|24.52|BrandD|
|   Clothing|25.23|BrandE|
|      Books|26.03|BrandB|
|       Toys| 28.5|BrandB|
|      Books|28.75|BrandC|
|       Toys|29.47|BrandE|
+-----------+-----+------+
only showing top 20 rows


In [ ]:
df.groupBy("Category").avg("Rating").show()

+-----------+------------------+
|   Category|       avg(Rating)|
+-----------+------------------+
|       Home|3.0842696629213497|
|     Sports| 2.995604395604396|
|Electronics| 2.969565217391304|
|   Clothing|3.0038461538461534|
|      Books| 3.124691358024692|
|       Toys|2.8065217391304342|
+-----------+------------------+



In [ ]:
df.groupBy("Brand").sum("Reviews") \
.orderBy(desc("sum(Reviews)")).show()

+------+------------+
| Brand|sum(Reviews)|
+------+------------+
|BrandA|       56656|
|BrandC|       53713|
|BrandE|       51547|
|BrandD|       51467|
|BrandB|       40990|
+------+------------+



In [ ]:
df.groupBy("Category").avg("Stock").show()

+-----------+------------------+
|   Category|        avg(Stock)|
+-----------+------------------+
|       Home| 58.28089887640449|
|     Sports| 51.53846153846154|
|Electronics| 47.04347826086956|
|   Clothing|51.833333333333336|
|      Books|48.592592592592595|
|       Toys| 52.82608695652174|
+-----------+------------------+



In [ ]:
df.groupBy("Category").avg("Discount").show()

+-----------+------------------+
|   Category|     avg(Discount)|
+-----------+------------------+
|       Home| 27.68719101123595|
|     Sports|22.075604395604394|
|Electronics|25.856956521739132|
|   Clothing| 22.57025641025641|
|      Books|26.518148148148146|
|       Toys|24.573260869565235|
+-----------+------------------+



In [ ]:
df.select("Category","Rating","Reviews","Demand_Score").show()

+-----------+------+-------+------------------+
|   Category|Rating|Reviews|      Demand_Score|
+-----------+------+-------+------------------+
|     Sports|   2.1|    228|             478.8|
|       Toys|   1.1|     95|104.50000000000001|
|Electronics|   3.6|    558|            2008.8|
|      Books|   4.0|    163|             652.0|
|   Clothing|   1.4|    389| 544.5999999999999|
|       Toys|   3.9|    549|            2141.1|
|      Books|   4.5|    370|            1665.0|
|Electronics|   2.2|     81|178.20000000000002|
|       Home|   1.7|    363|             617.1|
|     Sports|   3.5|    546|            1911.0|
|      Books|   3.8|    224| 851.1999999999999|
|Electronics|   2.1|    216|             453.6|
|       Home|   2.8|    271|             758.8|
|       Toys|   2.7|    597|            1611.9|
|   Clothing|   4.0|    881|            3524.0|
|     Sports|   2.5|    610|            1525.0|
|       Toys|   3.7|    117|432.90000000000003|
|      Books|   1.6|      3| 4.800000000